# 探針實驗:在 Colab 跑(長庚 MercuryAPI)

流程:**設定 → 連線測試 → 小量試跑 → 完整實驗 → 分析**。日誌存在 Google Drive,Colab 斷線後重新執行同一格就會**自動續跑**。

事前準備:
1. 到 MercuryAPI 領取金鑰(格式 `sk-mercury-…`)。左側 🔑「密鑰」新增 `MERCURY_API_KEY`,並開啟「筆記本存取權」。**不要把金鑰貼進程式格。**
2. 限制:**每分鐘 30 次、同時 2 個請求、每週 $10**。本筆記本已內建限速(每分鐘 27 次、同時 2 個),模型會**一個一個**跑,避免超過帳號整體上限。
3. 尖峰時段會排隊;大量執行請盡量排在晚上。
4. 注意:透過金鑰送出的請求內容會匿名保存供研究,不要放密碼或機密。

## 1. 安裝與取得程式

In [ ]:
import os, subprocess, sys
REPO = "Ting-liu0103/coop-llm"      # 改成你的 repo
tok = ""
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
url = f"https://{tok + '@' if tok else ''}github.com/{REPO}.git"
if not os.path.isdir("/content/coop-llm"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/content/coop-llm"], check=True)
else:
    subprocess.run(["git", "-C", "/content/coop-llm", "pull", "--ff-only"], check=True)
os.chdir("/content/coop-llm")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[openai,anthropic,plots,dev]"], check=True)
print("OK")

## 2. 自我檢查(78 項測試,不花錢)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(r.stdout[-400:])
assert r.returncode == 0, "測試沒全過,先不要往下跑"

## 3. 設定模型
兩個模型:一個商用小模型、一個開源模型(符合實驗設計)。
- `gpt-6-luna` 是否為最便宜的商用模型我**不確定**,可以改成別的(可用清單見「我的用量」頁)。試跑後看花費再決定。
- `gpt-oss-120b` 是開放權重模型。也可換成 gemma-4 / mistral-small-4 / nemotron。
- 這些模型可能先「思考」再回答,思考也佔 `max_tokens`。如果試跑時回覆常是空的或解析失敗,把 `max_tokens` 調大。

In [ ]:
import yaml
from google.colab import userdata

BASE = "https://air.cgu.edu.tw/mercuryapi/v1"
COMMON = dict(provider="openai_compat", base_url=BASE, api_key_env="MERCURY_API_KEY",
              rpm=27, max_concurrent=2, max_tokens=1200)
MODELS = [
    {"name": "api_small", "model": "gpt-6-luna", **COMMON},
    {"name": "open_model", "model": "gpt-oss-120b", **COMMON},
]
os.environ["MERCURY_API_KEY"] = userdata.get("MERCURY_API_KEY")
yaml.safe_dump({"models": MODELS}, open("configs/models_run.yaml", "w"), allow_unicode=True)
print([m["name"] for m in MODELS])

### 連線測試
對每個模型送一個最小請求。有些模型不接受自訂溫度(0.7);若被拒絕,這格會自動改成「不傳溫度」並更新設定。

In [ ]:
import asyncio, yaml
from coop_llm.llm import make_client

async def smoke(m):
    c = make_client(m)
    for temp in (0.7, None):
        try:
            r = await c.complete("你是測試員。", [{"role": "user", "content": '只回傳 JSON: {"ok": true}'}],
                                 temperature=temp, max_tokens=m.get("max_tokens", 1200))
            return temp, r.text[:80], r.tokens_in, r.tokens_out
        except Exception as e:
            print(f"  {m['name']} temperature={temp} 失敗: {type(e).__name__}: {str(e)[:160]}")
    return "ALL_FAILED", "", 0, 0

for m in MODELS:
    temp, text, tin, tout = await smoke(m)
    print(m["name"], "| temperature =", temp, "| 回覆:", repr(text), "| tokens", tin, tout)
    assert temp != "ALL_FAILED", f"{m['name']} 連不上,檢查金鑰、型號、網址"
    if temp is None:
        m["temperature"] = None   # 這個模型不傳溫度
yaml.safe_dump({"models": MODELS}, open("configs/models_run.yaml", "w"), allow_unicode=True)

## 4. 日誌存到 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
LOGS = "/content/drive/MyDrive/coop-llm-logs/probe"
os.makedirs(LOGS, exist_ok=True)
import glob
CONFIGS = sorted(glob.glob("configs/probe/*.yaml"))
print(CONFIGS)

## 5. 小量試跑(每格 2 場)
先確認格式正常、算出費用。跑完看下一格的解析失敗率與 token 用量。

In [ ]:
def run_all(reps, out, start_rep=0):
    # 模型一個一個跑:帳號的 30 RPM / 同時 2 個是「整個帳號」共用的,兩個模型同時跑會互相搶
    for m in MODELS:
        for c in CONFIGS:
            r = subprocess.run([sys.executable, "-m", "coop_llm.runner", "--condition", c, "--models",
                                "configs/models_run.yaml", "--only-model", m["name"], "--reps", str(reps),
                                "--start-rep", str(start_rep), "--out", out, "--concurrency", "2"],
                               capture_output=True, text=True)
            tail = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
            print(m["name"], c.split("/")[-1], "→", tail)

PILOT = "/content/drive/MyDrive/coop-llm-logs/pilot"
# 試跑前:先到「我的用量」記下目前「已用 $」,跑完再看一次,差額 ÷ 場次 = 每場成本
run_all(2, PILOT)

In [ ]:
import pandas as pd
from coop_llm.analysis import load_logs, parse_failure_report
rounds, chats, summary = load_logs(PILOT)
print("完成場次:", summary["game_id"].nunique(), "(這次試跑預期:每個模型 23 格 × 3 賽局 × 2 場 = 138 場)")
print(parse_failure_report(rounds).to_string())
print("輸入 token 合計:", int(rounds["tokens_in"].sum()), " 輸出 token 合計:", int(rounds["tokens_out"].sum()))
print("輸出 token 接近 max_tokens 的比例:", round((rounds["tokens_out"] >= 1100).mean(), 3), "← 偏高代表被截斷")
print("看一段想法:", rounds["thought"].dropna().iloc[0][:200])

**檢查清單再往下:**
- 解析失敗率(上表 `fail_rate`)明顯 > 5%,或輸出 token 常頂到上限 → 先調大 `max_tokens` 或調提示詞,不要放大。
- **算費用**:到「我的用量」看已用 $ 的增加量,÷ 試跑場數 × 完整場數(每格 20 場)= 完整實驗費用。每週額度 $10,超過就分週跑(下週一 00:00 重置)。
- **算時間**:完整實驗上限約 3.1 萬次呼叫 ÷ 每分鐘 27 次 ≈ 19 小時。分幾天、挑晚上跑;每次斷線後重跑 `run_all` 即續跑。
- 想法內容看起來合理(沒有空白、沒有亂碼)。

## 6. 完整實驗(每格 20 場,可重複執行續跑)

In [ ]:
run_all(20, LOGS)

## 7. 分析

In [ ]:
out = subprocess.run([sys.executable, "-m", "coop_llm.analyze_probe", "--logs", LOGS,
                      "--out", "/content/drive/MyDrive/coop-llm-logs/results"], capture_output=True, text=True)
print(out.stdout[-600:], out.stderr[-600:])

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/coop-llm-logs/results/quality_map_*.png")):
    display(Image(p))

結果檔都在 Drive 的 `coop-llm-logs/results/`(`di_pairs.csv`、`comparisons_*.csv`、圖)。